In [15]:
# Filter French/Kabyle pairs based on length difference
# The file has ~1.2M pairs, each line: French<TABS>Kabyle (separated by multiple tabs)

import os
import re

INPUT_FILE = r"d:\Documents\py\projects\chat_aqvayli\all_fr_kab_pairs.txt"
OUTPUT_FILE = r"d:\Documents\py\projects\chat_aqvayli\filtered_fr_kab_pairs.txt"

# Configuration: Maximum allowed length ratio between sentences
# If one sentence is more than X times longer than the other, remove the pair
# e.g., French can be at most 1.5x longer than Kabyle and vice versa
MAX_LENGTH_RATIO = 1.5

# Alternative: Maximum character difference
MAX_CHAR_DIFFERENCE = 100  # Maximum absolute difference in characters

# Choose filtering method: "ratio" or "difference"
FILTER_METHOD = "ratio"

# Separator pattern: one or more tabs
TAB_PATTERN = re.compile(r'\t+')

In [21]:
# Analyze the data first - sample some pairs to understand length distribution
sample_pairs = []
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 1000:  # Sample first 1000 pairs
            break
        line = line.strip()
        # Split by one or more tabs
        parts = TAB_PATTERN.split(line)

        if len(parts) >= 2:
            fr, kab = parts[0].strip(), parts[1].strip()
            if fr and kab:  # Only add non-empty pairs
                sample_pairs.append((fr, kab, len(fr), len(kab)))

# Debug: show first few raw pairs loaded
print("First 3 pairs loaded:")
for p in sample_pairs[:3]:
    print(f"  FR({len(p[0])}): '{p[0][:50]}'")
    print(f"  KAB({len(p[1])}): '{p[1][:50]}'")
    print()

# Calculate statistics
ratios = []
differences = []
for fr, kab, len_fr, len_kab in sample_pairs:
    if len_kab > 0 and len_fr > 0:
        ratio = max(len_fr, len_kab) / min(len_fr, len_kab)
        diff = abs(len_fr - len_kab)
        ratios.append(ratio)
        differences.append(diff)

print(f"Sample size: {len(sample_pairs)} pairs")
if ratios:
    print(f"\nLength Ratio (max/min) Statistics:")
    print(f"  Min ratio: {min(ratios):.2f}")
    print(f"  Max ratio: {max(ratios):.2f}")
    print(f"  Avg ratio: {sum(ratios)/len(ratios):.2f}")
    print(f"\nCharacter Difference Statistics:")
    print(f"  Min diff: {min(differences)}")
    print(f"  Max diff: {max(differences)}")
    print(f"  Avg diff: {sum(differences)/len(differences):.1f}")
else:
    print("No valid pairs found! Check the file format.")

First 3 pairs loaded:
  FR(27): 'Nous avons beaucoup marché.'
  KAB(13): 'Aṭas i nelḥa.'

  FR(27): 'Il est venu plusieurs fois.'
  KAB(21): 'Yusa-d aṭas n tikkal.'

  FR(17): 'Il parle anglais.'
  KAB(20): 'Yettmeslay taglizit.'

Sample size: 1000 pairs

Length Ratio (max/min) Statistics:
  Min ratio: 1.00
  Max ratio: 4.00
  Avg ratio: 1.34

Character Difference Statistics:
  Min diff: 0
  Max diff: 51
  Avg diff: 6.5


In [22]:
# Show examples of pairs with extreme length differences
print("Examples of pairs with HIGH length ratio (potentially bad pairs):\n")
extreme_pairs = sorted(sample_pairs, key=lambda x: max(
    x[2], x[3]) / max(min(x[2], x[3]), 1), reverse=True)

for fr, kab, len_fr, len_kab in extreme_pairs[:10]:
    ratio = max(len_fr, len_kab) / max(min(len_fr, len_kab), 1)
    print(f"Ratio: {ratio:.2f} | FR({len_fr}): {fr}")
    print(f"           | KAB({len_kab}): {kab}")
    print("-" * 80)

Examples of pairs with HIGH length ratio (potentially bad pairs):

Ratio: 4.00 | FR(16): Agenouille-toi !
           | KAB(4): Knu!
--------------------------------------------------------------------------------
Ratio: 3.83 | FR(23): Est-ce que c'est cuit ?
           | KAB(6): Yewwa?
--------------------------------------------------------------------------------
Ratio: 3.60 | FR(18): Sans aucun doute !
           | KAB(5): Iban!
--------------------------------------------------------------------------------
Ratio: 3.60 | FR(18): C'est insuffisant.
           | KAB(5): Drus.
--------------------------------------------------------------------------------
Ratio: 3.20 | FR(16): Agenouille-toi !
           | KAB(5): Anez!
--------------------------------------------------------------------------------
Ratio: 2.90 | FR(29): Elle est en train de pleurer.
           | KAB(10): La tettru.
--------------------------------------------------------------------------------
Ratio: 2.80 | FR(14):

In [ ]:
# Show examples of pairs with GOOD length ratio
print("Examples of pairs with LOW length ratio (good pairs):\n")
good_pairs = sorted(sample_pairs, key=lambda x: max(
    x[2], x[3]) / max(min(x[2], x[3]), 1))

for fr, kab, len_fr, len_kab in good_pairs[:10]:
    ratio = max(len_fr, len_kab) / max(min(len_fr, len_kab), 1)
    print(f"Ratio: {ratio:.2f} | FR({len_fr}): {fr}")
    print(f"           | KAB({len_kab}): {kab}")
    print("-" * 80)

In [ ]:
# Function to filter pairs based on chosen method
def should_keep_pair(fr_text: str, kab_text: str, method: str = "ratio",
                     max_ratio: float = 3.0, max_diff: int = 100) -> bool:
    """
    Returns True if the pair should be kept, False if it should be removed.

    Methods:
    - "ratio": Keep if length ratio is <= max_ratio
    - "difference": Keep if absolute character difference is <= max_diff
    - "both": Keep only if both conditions are satisfied
    """
    len_fr = len(fr_text.strip())
    len_kab = len(kab_text.strip())

    # Skip empty sentences
    if len_fr == 0 or len_kab == 0:
        return False

    ratio = max(len_fr, len_kab) / min(len_fr, len_kab)
    diff = abs(len_fr - len_kab)

    if method == "ratio":
        return ratio <= max_ratio
    elif method == "difference":
        return diff <= max_diff
    elif method == "both":
        return ratio <= max_ratio and diff <= max_diff
    else:
        return True


# Test the function
print("Testing filter function:")
test_pairs = [
    ("Bonjour", "Azul"),  # Good
    # Bad - big difference
    ("Je suis très content de vous rencontrer aujourd'hui", "Azul"),
    ("Oui", "Ih"),  # Good
]
for fr, kab in test_pairs:
    keep = should_keep_pair(fr, kab, method="ratio", max_ratio=3.0)
    ratio = max(len(fr), len(kab)) / min(len(fr), len(kab))
    print(f"Keep: {keep} | Ratio: {ratio:.2f} | FR: '{fr}' | KAB: '{kab}'")

In [ ]:
# Process the full file and filter pairs
# This may take a few minutes for 1.2M lines

def filter_pairs_file(input_file: str, output_file: str,
                      method: str = "ratio", max_ratio: float = 3.0, max_diff: int = 100):
    """
    Read input file, filter pairs, and write to output file.
    Returns statistics about the filtering process.
    """
    total_lines = 0
    kept_lines = 0
    removed_lines = 0
    malformed_lines = 0

    with open(input_file, 'r', encoding='utf-8') as fin, \
            open(output_file, 'w', encoding='utf-8') as fout:

        for line in fin:
            total_lines += 1

            # Progress indicator every 100k lines
            if total_lines % 100000 == 0:
                print(f"Processed {total_lines:,} lines...")

            line = line.strip()
            parts = TAB_PATTERN.split(line)  # Use the tab pattern from cell 1

            if len(parts) < 2:
                malformed_lines += 1
                continue

            fr_text = parts[0].strip()
            kab_text = parts[1].strip()

            if should_keep_pair(fr_text, kab_text, method=method,
                                max_ratio=max_ratio, max_diff=max_diff):
                fout.write(f"{fr_text}\t{kab_text}\n")
                kept_lines += 1
            else:
                removed_lines += 1

    return {
        'total': total_lines,
        'kept': kept_lines,
        'removed': removed_lines,
        'malformed': malformed_lines,
        'kept_percentage': (kept_lines / total_lines * 100) if total_lines > 0 else 0
    }


print("Filter function ready. Run the next cell to process the file.")

In [ ]:
# ⚠️ RUN THIS CELL TO FILTER THE FILE
# Adjust parameters as needed before running

# PARAMETERS - Adjust these before running:
# Maximum ratio allowed (e.g., 2.5 means one can be at most 2.5x longer)
MAX_LENGTH_RATIO = 2.5
# Maximum character difference (used if method="difference")
MAX_CHAR_DIFF = 80
FILTER_METHOD = "ratio"   # Options: "ratio", "difference", "both"

print(
    f"Starting filter with method='{FILTER_METHOD}', max_ratio={MAX_LENGTH_RATIO}, max_diff={MAX_CHAR_DIFF}")
print(f"Input: {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print("-" * 50)

stats = filter_pairs_file(
    INPUT_FILE,
    OUTPUT_FILE,
    method=FILTER_METHOD,
    max_ratio=MAX_LENGTH_RATIO,
    max_diff=MAX_CHAR_DIFF
)

print("\n" + "=" * 50)
print("FILTERING COMPLETE")
print("=" * 50)
print(f"Total pairs processed: {stats['total']:,}")
print(
    f"Pairs kept:           {stats['kept']:,} ({stats['kept_percentage']:.1f}%)")
print(f"Pairs removed:        {stats['removed']:,}")
print(f"Malformed lines:      {stats['malformed']:,}")
print(f"\nOutput saved to: {OUTPUT_FILE}")

In [ ]:
# Verify the output - sample some filtered pairs
print("Sample of KEPT pairs from the filtered file:\n")
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            fr, kab = parts[0], parts[1]
            ratio = max(len(fr), len(kab)) / min(len(fr), len(kab))
            print(f"Ratio: {ratio:.2f} | FR({len(fr)}): {fr[:50]}")
            print(f"           | KAB({len(kab)}): {kab[:50]}")
            print("-" * 60)